In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Enable GPU in Kaggle Settings."
    )

print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
from pathlib import Path
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoImageProcessor,
    AutoTokenizer,
    SiglipModel,
    SiglipForImageClassification
)

import torch.nn as nn

print("Imports successful.")

Imports successful.


In [3]:
INPUT_ROOT = Path("/kaggle/input")

dataset_candidates = [
    p for p in INPUT_ROOT.rglob("kaggle_dataset")
    if p.is_dir()
]

print("Dataset candidates:")

for p in dataset_candidates:
    print(p)

if not dataset_candidates:
    raise FileNotFoundError(
        "kaggle_dataset not found. "
        "Attach GeoSigLIP DCID-7 Lithology Dataset."
    )

DATA_ROOT = dataset_candidates[0]

print("\nUsing:")
print(DATA_ROOT)

Dataset candidates:
/kaggle/input/datasets/utkarshrode/geosiglip-dcid-7-lithology-dataset/kaggle_dataset

Using:
/kaggle/input/datasets/utkarshrode/geosiglip-dcid-7-lithology-dataset/kaggle_dataset


In [4]:
required_files = [
    "train.csv",
    "validation.csv",
    "test.csv"
]

for filename in required_files:

    path = DATA_ROOT / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}"
        )

IMAGE_ROOT = DATA_ROOT / "images"

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Missing images directory: {IMAGE_ROOT}"
    )

print("Dataset structure verified.")

Dataset structure verified.


In [5]:
test_df = pd.read_csv(
    DATA_ROOT / "test.csv"
)

print(
    "Test images:",
    len(test_df)
)

print("\nTest class distribution:")

print(
    test_df["label"]
    .value_counts()
    .sort_index()
)

Test images: 1750

Test class distribution:
label
Basalt             250
Granite            250
Gray siltstone     250
Light sandstone    250
Marble             250
Mudstone           250
Red sandstone      250
Name: count, dtype: int64


In [6]:
test_df["image_path"] = test_df[
    "image_path"
].apply(
    lambda x: DATA_ROOT / str(x)
)

missing = test_df[
    "image_path"
].apply(
    lambda x: not x.exists()
)

print(
    "Missing images:",
    missing.sum()
)

if missing.any():

    print(
        test_df.loc[
            missing,
            "image_path"
        ].head(10)
    )

    raise FileNotFoundError(
        "Some test images are missing."
    )

print("All 1,750 test images found.")

Missing images: 0
All 1,750 test images found.


In [7]:
CLASS_NAMES = [
    "Red sandstone",
    "Light sandstone",
    "Gray siltstone",
    "Mudstone",
    "Granite",
    "Basalt",
    "Marble"
]

LABEL2ID = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

ID2LABEL = {
    i: name
    for i, name in enumerate(CLASS_NAMES)
}

NUM_CLASSES = len(CLASS_NAMES)

print(LABEL2ID)

{'Red sandstone': 0, 'Light sandstone': 1, 'Gray siltstone': 2, 'Mudstone': 3, 'Granite': 4, 'Basalt': 5, 'Marble': 6}


In [8]:
MODEL_NAME = "google/siglip-base-patch16-224"

image_processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

print("Image processor loaded.")

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Image processor loaded.


In [9]:
class LithologyTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        processor
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        encoded = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": encoded[
                "pixel_values"
            ].squeeze(0),

            "label": torch.tensor(
                LABEL2ID[row["label"]],
                dtype=torch.long
            )
        }

In [10]:
test_dataset = LithologyTestDataset(
    test_df,
    image_processor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(
    "Test samples:",
    len(test_dataset)
)

print(
    "Test batches:",
    len(test_loader)
)

Test samples: 1750
Test batches: 110


In [11]:
batch = next(iter(test_loader))

print(
    "Pixel values:",
    batch["pixel_values"].shape
)

print(
    "Labels:",
    batch["label"].shape
)

Pixel values: torch.Size([16, 3, 224, 224])
Labels: torch.Size([16])


In [12]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("All mounted files containing 'best_model' or 'geosiglip':\n")

found = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        full_path = str(p).lower()

        if (
            "best_model" in name
            or "geosiglip" in full_path
        ):
            found.append(p)

for p in found:
    print(p)

if not found:
    print("No checkpoint-related file found.")

All mounted files containing 'best_model' or 'geosiglip':

/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/.format_version
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/.storage_alignment
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data.pkl
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/version
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/byteorder
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/.data/serialization_id
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data/248
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data/7
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data/135
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data/47
/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint/best_model/data/183
/kaggle/input/datasets

In [13]:
class LoRALinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank=16,
        alpha=32,
        dropout=0.05
    ):
        super().__init__()

        if not isinstance(
            base_layer,
            nn.Linear
        ):
            raise TypeError(
                "Expected nn.Linear."
            )

        self.base_layer = base_layer

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Freeze pretrained layer
        for parameter in (
            self.base_layer.parameters()
        ):
            parameter.requires_grad = False

        layer_device = (
            base_layer.weight.device
        )

        layer_dtype = (
            base_layer.weight.dtype
        )

        self.lora_A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
            device=layer_device,
            dtype=layer_dtype
        )

        self.lora_B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
            device=layer_device,
            dtype=layer_dtype
        )

        self.dropout = nn.Dropout(
            dropout
        )

        nn.init.kaiming_uniform_(
            self.lora_A.weight,
            a=np.sqrt(5)
        )

        nn.init.zeros_(
            self.lora_B.weight
        )

    def forward(self, x):

        base_output = self.base_layer(x)

        lora_output = self.lora_B(
            self.lora_A(
                self.dropout(x)
            )
        )

        return (
            base_output
            + self.scaling * lora_output
        )

In [14]:
fine_tuned_model = (
    SiglipForImageClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True
    )
    .to(device)
)

# Freeze everything first
for parameter in fine_tuned_model.parameters():
    parameter.requires_grad = False


# Recreate the exact LoRA layers
for layer in (
    fine_tuned_model
    .vision_model
    .encoder
    .layers
):

    layer.self_attn.q_proj = LoRALinear(
        layer.self_attn.q_proj,
        rank=16,
        alpha=32,
        dropout=0.05
    )

    layer.self_attn.v_proj = LoRALinear(
        layer.self_attn.v_proj,
        rank=16,
        alpha=32,
        dropout=0.05
    )


# Make LoRA parameters trainable
for layer in (
    fine_tuned_model
    .vision_model
    .encoder
    .layers
):

    for parameter in (
        layer.self_attn.q_proj
        .lora_A.parameters()
    ):
        parameter.requires_grad = True

    for parameter in (
        layer.self_attn.q_proj
        .lora_B.parameters()
    ):
        parameter.requires_grad = True

    for parameter in (
        layer.self_attn.v_proj
        .lora_A.parameters()
    ):
        parameter.requires_grad = True

    for parameter in (
        layer.self_attn.v_proj
        .lora_B.parameters()
    ):
        parameter.requires_grad = True


# Classifier was trained too
for parameter in (
    fine_tuned_model.classifier.parameters()
):
    parameter.requires_grad = True


fine_tuned_model = fine_tuned_model.to(device)
fine_tuned_model.eval()

print(
    "Fine-tuned architecture reconstructed."
)

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipForImageClassification LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |

Fine-tuned architecture reconstructed.


In [15]:
from pathlib import Path
import torch

# Find the checkpoint directly
checkpoint_candidates = [
    p
    for p in Path("/kaggle/input").rglob("best_model.pt")
    if p.is_file()
]

print("Checkpoint candidates:")

for p in checkpoint_candidates:
    print(p)

if not checkpoint_candidates:
    raise FileNotFoundError(
        "best_model.pt not found. "
        "Make sure GeoSigLIP LoRA Checkpoint is attached."
    )

CHECKPOINT_PATH = checkpoint_candidates[0]

print("\nUsing checkpoint:")
print(CHECKPOINT_PATH)

print(
    "Size:",
    round(
        CHECKPOINT_PATH.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

print("\nCheckpoint loaded successfully.")
print(
    "Number of tensors:",
    len(checkpoint)
)

Checkpoint candidates:


FileNotFoundError: best_model.pt not found. Make sure GeoSigLIP LoRA Checkpoint is attached.

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Top-level inputs:\n")

for p in root.iterdir():
    print(p)

print("\nPossible checkpoint files:\n")

for p in root.rglob("*"):
    if p.is_file() and (
        "best_model" in p.name.lower()
        or "checkpoint" in p.name.lower()
        or p.suffix.lower() in [".pt", ".pth", ".zip"]
    ):
        print(p)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input/datasets").iterdir():
    print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets/utkarshrode")

print("Inside /kaggle/input/datasets/utkarshrode:\n")

for p in root.iterdir():
    print(p)

In [ ]:
from pathlib import Path

CHECKPOINT_ROOT = Path(
    "/kaggle/input/datasets/utkarshrode/geosiglip-lora-checkpoint"
)

print("Checkpoint dataset contents:")

for p in CHECKPOINT_ROOT.rglob("*"):
    if p.is_file():
        print(p)

In [ ]:
CHECKPOINT_PATH = next(
    CHECKPOINT_ROOT.rglob("best_model.pt")
)

print("CHECKPOINT_PATH:")
print(CHECKPOINT_PATH)

print(
    "Size:",
    round(
        CHECKPOINT_PATH.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/utkarshrode/"
    "geosiglip-lora-checkpoint/best_model"
)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint directory not found:\n"
        f"{CHECKPOINT_PATH}"
    )

print("Checkpoint found:")
print(CHECKPOINT_PATH)

print("\nCheckpoint contents:")
for p in CHECKPOINT_PATH.iterdir():
    print(p.name)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

class LoRALinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank=16,
        alpha=32,
        dropout=0.05
    ):
        super().__init__()

        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        for p in self.base_layer.parameters():
            p.requires_grad = False

        device_ = base_layer.weight.device
        dtype_ = base_layer.weight.dtype

        self.lora_A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
            device=device_,
            dtype=dtype_
        )

        self.lora_B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
            device=device_,
            dtype=dtype_
        )

        self.dropout = nn.Dropout(dropout)

        nn.init.kaiming_uniform_(
            self.lora_A.weight,
            a=np.sqrt(5)
        )

        nn.init.zeros_(
            self.lora_B.weight
        )

    def forward(self, x):

        return (
            self.base_layer(x)
            + self.scaling
            * self.lora_B(
                self.lora_A(
                    self.dropout(x)
                )
            )
        )

print("LoRALinear defined.")

In [ ]:
fine_tuned_model = (
    SiglipForImageClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True
    )
    .to(device)
)

# Freeze everything
for p in fine_tuned_model.parameters():
    p.requires_grad = False

# Recreate LoRA exactly as in Notebook 03
for layer in fine_tuned_model.vision_model.encoder.layers:

    layer.self_attn.q_proj = LoRALinear(
        layer.self_attn.q_proj,
        rank=16,
        alpha=32,
        dropout=0.05
    )

    layer.self_attn.v_proj = LoRALinear(
        layer.self_attn.v_proj,
        rank=16,
        alpha=32,
        dropout=0.05
    )

# Enable LoRA parameters
for layer in fine_tuned_model.vision_model.encoder.layers:

    for p in layer.self_attn.q_proj.lora_A.parameters():
        p.requires_grad = True

    for p in layer.self_attn.q_proj.lora_B.parameters():
        p.requires_grad = True

    for p in layer.self_attn.v_proj.lora_A.parameters():
        p.requires_grad = True

    for p in layer.self_attn.v_proj.lora_B.parameters():
        p.requires_grad = True

# Enable classifier
for p in fine_tuned_model.classifier.parameters():
    p.requires_grad = True

fine_tuned_model = fine_tuned_model.to(device)

print("Exact fine-tuned architecture reconstructed.")

In [ ]:
import zipfile
from pathlib import Path

CHECKPOINT_ROOT = Path(
    "/kaggle/input/datasets/utkarshrode/"
    "geosiglip-lora-checkpoint/best_model"
)

REBUILT_CHECKPOINT = Path(
    "/kaggle/working/best_model.pt"
)

if not CHECKPOINT_ROOT.exists():
    raise FileNotFoundError(
        f"Checkpoint directory not found:\n{CHECKPOINT_ROOT}"
    )

with zipfile.ZipFile(
    REBUILT_CHECKPOINT,
    "w",
    compression=zipfile.ZIP_STORED
) as zf:

    for file_path in CHECKPOINT_ROOT.rglob("*"):

        if file_path.is_file():

            archive_name = file_path.relative_to(
                CHECKPOINT_ROOT
            )

            zf.write(
                file_path,
                arcname=str(archive_name)
            )

print(
    "Rebuilt checkpoint:",
    REBUILT_CHECKPOINT
)

print(
    "Size:",
    round(
        REBUILT_CHECKPOINT.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
# Cell 16 — Reconstruct and load the PyTorch checkpoint correctly

import zipfile
from pathlib import Path
import torch

CHECKPOINT_ROOT = Path(
    "/kaggle/input/datasets/utkarshrode/"
    "geosiglip-lora-checkpoint/best_model"
)

REBUILT_CHECKPOINT = Path(
    "/kaggle/working/best_model.pt"
)

# Recreate the original PyTorch ZIP structure.
# All checkpoint files must live under one root folder.
with zipfile.ZipFile(
    REBUILT_CHECKPOINT,
    "w",
    compression=zipfile.ZIP_STORED
) as zf:

    for file_path in CHECKPOINT_ROOT.rglob("*"):

        if file_path.is_file():

            relative_path = file_path.relative_to(
                CHECKPOINT_ROOT
            )

            archive_path = Path(
                "best_model"
            ) / relative_path

            zf.write(
                file_path,
                arcname=str(archive_path)
            )

print(
    "Rebuilt checkpoint:",
    REBUILT_CHECKPOINT
)

print(
    "Size:",
    round(
        REBUILT_CHECKPOINT.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

# Load checkpoint
checkpoint = torch.load(
    REBUILT_CHECKPOINT,
    map_location=device,
    weights_only=True
)

print(
    "Checkpoint loaded successfully."
)

print(
    "Number of tensors:",
    len(checkpoint)
)

# Load into reconstructed model
result = fine_tuned_model.load_state_dict(
    checkpoint,
    strict=True
)

print(
    "Missing keys:",
    len(result.missing_keys)
)

print(
    "Unexpected keys:",
    len(result.unexpected_keys)
)

if (
    len(result.missing_keys) == 0
    and len(result.unexpected_keys) == 0
):
    print(
        "Checkpoint matches model exactly."
    )
else:
    raise RuntimeError(
        "Checkpoint does not exactly match the model."
    )

fine_tuned_model.eval()

print("Fine-tuned model ready for test evaluation.")

In [ ]:
batch = next(iter(test_loader))

pixel_values = batch[
    "pixel_values"
].to(device)

with torch.no_grad():

    outputs = fine_tuned_model(
        pixel_values=pixel_values
    )

print(
    "Logits shape:",
    outputs.logits.shape
)

print(
    "Predictions:",
    torch.argmax(
        outputs.logits,
        dim=1
    )[:10].cpu().tolist()
)

In [ ]:
def predict_finetuned(
    model,
    loader
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_confidences = []

    with torch.no_grad():

        for batch in loader:

            pixel_values = batch[
                "pixel_values"
            ].to(device)

            labels = batch[
                "label"
            ]

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            confidences, predictions = (
                probabilities.max(dim=1)
            )

            all_labels.extend(
                labels.tolist()
            )

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_confidences.extend(
                confidences.cpu().tolist()
            )

    return (
        np.array(all_labels),
        np.array(all_predictions),
        np.array(all_confidences)
    )


ft_labels, ft_predictions, ft_confidences = (
    predict_finetuned(
        fine_tuned_model,
        test_loader
    )
)

print(
    "Test predictions:",
    len(ft_predictions)
)

In [ ]:
def calculate_metrics(
    labels,
    predictions
):

    accuracy = np.mean(
        labels == predictions
    )

    per_class = []

    for class_id in range(NUM_CLASSES):

        tp = np.sum(
            (labels == class_id)
            &
            (predictions == class_id)
        )

        fp = np.sum(
            (labels != class_id)
            &
            (predictions == class_id)
        )

        fn = np.sum(
            (labels == class_id)
            &
            (predictions != class_id)
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_class.append({
            "class": CLASS_NAMES[class_id],
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    return {
        "accuracy": accuracy,
        "macro_precision": np.mean([
            x["precision"]
            for x in per_class
        ]),
        "macro_recall": np.mean([
            x["recall"]
            for x in per_class
        ]),
        "macro_f1": np.mean([
            x["f1"]
            for x in per_class
        ]),
        "per_class": per_class
    }

In [ ]:
ft_metrics = calculate_metrics(
    ft_labels,
    ft_predictions
)

print("=" * 60)
print("SIGLIP + LoRA — FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Accuracy:  "
    f"{ft_metrics['accuracy'] * 100:.3f}%"
)

print(
    f"Precision: "
    f"{ft_metrics['macro_precision'] * 100:.3f}%"
)

print(
    f"Recall:    "
    f"{ft_metrics['macro_recall'] * 100:.3f}%"
)

print(
    f"Macro F1:  "
    f"{ft_metrics['macro_f1'] * 100:.3f}%"
)

In [ ]:
ft_per_class_df = pd.DataFrame(
    ft_metrics["per_class"]
)

display(ft_per_class_df)

In [ ]:
ft_cm = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=int
)

for actual, predicted in zip(
    ft_labels,
    ft_predictions
):

    ft_cm[
        actual,
        predicted
    ] += 1

print(ft_cm)

In [ ]:
plt.figure(figsize=(9, 8))

plt.imshow(ft_cm)

plt.xticks(
    range(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(NUM_CLASSES),
    CLASS_NAMES
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title(
    "SigLIP + LoRA — Test Confusion Matrix"
)

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            ft_cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

In [ ]:
zero_shot_model = (
    SiglipModel
    .from_pretrained(
        MODEL_NAME
    )
    .to(device)
)

zero_shot_model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Zero-shot SigLIP loaded.")

In [ ]:
PROMPTS = [
    f"a geological photograph of {name.lower()} rock"
    for name in CLASS_NAMES
]

for i, prompt in enumerate(PROMPTS):
    print(
        f"{i}: {prompt}"
    )

In [ ]:
text_outputs = zero_shot_model.get_text_features(
    **text_inputs
)

text_features = text_outputs.pooler_output

text_features = (
    text_features
    / text_features.norm(
        dim=-1,
        keepdim=True
    )
)

print(
    "Text features:",
    text_features.shape
)

In [ ]:
def predict_zero_shot(
    model,
    loader,
    text_features
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_confidences = []

    with torch.no_grad():

        logit_scale = model.logit_scale.exp()
        logit_bias = model.logit_bias

        for batch in loader:

            pixel_values = batch[
                "pixel_values"
            ].to(device)

            image_outputs = model.get_image_features(
                pixel_values=pixel_values
            )

            image_features = (
                image_outputs.pooler_output
            )

            image_features = (
                image_features
                / image_features.norm(
                    dim=-1,
                    keepdim=True
                )
            )

            logits = (
                image_features
                @ text_features.T
            )

            logits = (
                logits
                * logit_scale
                + logit_bias
            )

            probabilities = torch.sigmoid(
                logits
            )

            confidences, predictions = (
                probabilities.max(dim=1)
            )

            all_labels.extend(
                batch["label"].tolist()
            )

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_confidences.extend(
                confidences.cpu().tolist()
            )

    return (
        np.array(all_labels),
        np.array(all_predictions),
        np.array(all_confidences)
    )

In [ ]:
# Cell 28 — Run zero-shot evaluation + calculate metrics

zs_labels, zs_predictions, zs_confidences = (
    predict_zero_shot(
        zero_shot_model,
        test_loader,
        text_features
    )
)

zs_metrics = calculate_metrics(
    zs_labels,
    zs_predictions
)

print("=" * 60)
print("SIGLIP ZERO-SHOT — FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Accuracy:  "
    f"{zs_metrics['accuracy'] * 100:.3f}%"
)

print(
    f"Precision: "
    f"{zs_metrics['macro_precision'] * 100:.3f}%"
)

print(
    f"Recall:    "
    f"{zs_metrics['macro_recall'] * 100:.3f}%"
)

print(
    f"Macro F1:  "
    f"{zs_metrics['macro_f1'] * 100:.3f}%"
)

print(
    "\nTest predictions:",
    len(zs_predictions)
)

In [ ]:
comparison_df = pd.DataFrame({

    "Model": [
        "SigLIP Zero-Shot",
        "SigLIP + LoRA"
    ],

    "Accuracy": [
        zs_metrics["accuracy"],
        ft_metrics["accuracy"]
    ],

    "Macro Precision": [
        zs_metrics["macro_precision"],
        ft_metrics["macro_precision"]
    ],

    "Macro Recall": [
        zs_metrics["macro_recall"],
        ft_metrics["macro_recall"]
    ],

    "Macro F1": [
        zs_metrics["macro_f1"],
        ft_metrics["macro_f1"]
    ]
})

display(comparison_df)

In [ ]:
accuracy_improvement = (
    ft_metrics["accuracy"]
    - zs_metrics["accuracy"]
)

f1_improvement = (
    ft_metrics["macro_f1"]
    - zs_metrics["macro_f1"]
)

print(
    f"Accuracy improvement: "
    f"{accuracy_improvement * 100:.3f} "
    "percentage points"
)

print(
    f"Macro F1 improvement: "
    f"{f1_improvement * 100:.3f} "
    "percentage points"
)

In [ ]:
RESULTS_DIR = Path(
    "/kaggle/working/geosiglip-final-results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_df.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)

ft_per_class_df.to_csv(
    RESULTS_DIR / "finetuned_per_class.csv",
    index=False
)

pd.DataFrame(
    zs_metrics["per_class"]
).to_csv(
    RESULTS_DIR / "zero_shot_per_class.csv",
    index=False
)

pd.DataFrame(
    ft_cm
).to_csv(
    RESULTS_DIR / "finetuned_confusion_matrix.csv",
    index=False
)

pd.DataFrame(
    zs_labels
).to_csv(
    RESULTS_DIR / "zero_shot_labels.csv",
    index=False
)

print(
    "All final evaluation results saved to:"
)

print(RESULTS_DIR)

In [ ]:
print("=" * 70)
print("GEOSIGLIP — FINAL EVALUATION COMPLETE")
print("=" * 70)

print(
    "\nTest set:",
    len(test_df),
    "images"
)

print("\nZERO-SHOT SIGLIP")
print(
    f"Accuracy: "
    f"{zs_metrics['accuracy'] * 100:.3f}%"
)

print(
    f"Macro F1: "
    f"{zs_metrics['macro_f1'] * 100:.3f}%"
)

print("\nSIGLIP + LoRA")
print(
    f"Accuracy: "
    f"{ft_metrics['accuracy'] * 100:.3f}%"
)

print(
    f"Macro F1: "
    f"{ft_metrics['macro_f1'] * 100:.3f}%"
)

print("\nIMPROVEMENT")
print(
    f"Accuracy: "
    f"+{accuracy_improvement * 100:.3f} pp"
)

print(
    f"Macro F1: "
    f"+{f1_improvement * 100:.3f} pp"
)

print("\nResults directory:")
print(RESULTS_DIR)